# lncRNA and NS-Forest: Downsampled Coding Set Draft

This notebook contains a draft workflow to analyze the performance of an `.h5ad` dataset in terms of its full gene set and lncRNA gene subset.

**Note:** The input data file should be a single-cell RNA-seq dataset in h5ad format, with cell type annotations in the obs dataframe and gene annotations in the var dataframe.

**Note:** If using GitHub for version control and repository sharing, ensure that you add the path to your data folder to the repository's `.gitignore` file, to prevent yourself from exceeding the GitHub's storage limits.

## Unix Command Structure

`python nsforest_coding_downsample.py {data file name} {tissue} {cluster header} {subset col}`

## Setup

In [20]:
# libraries
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import random
import scanpy as sc
import anndata as ad
import plotly.io as pio
pio.renderers.default = "notebook"
import nsforest as ns
import io
import contextlib
import warnings

In [2]:
## HEADER FUNCTION!
def print_header(title):
    width = 20
    print("\n" + "=" * width)
    print(f"{title.upper():^{width}}")
    print("=" * width)

In [ ]:
# header references
cluster_headers = {"bm" : "Curated_annotation",
                  "breast" : "level2",
                  "kidney" : "subclass.full",
                  "liver" : "Curated_annotation",
                  "lung" : "ann_finest_level",
                  "m1" : "cluster_id",
                  "mtg" : "cluster_id",
                  "retina" : "author_cell_type",
                  "spc" : "ThirdAnnotation"
                  }

genetype_headers = {"bm" : "feature_type",
                   "breast" : "feature_type",
                   "kidney" : "feature_type",
                   "liver" : "feature_type",
                   "lung" : "feature_type",
                   "m1" : "biotype",
                   "mtg" : "biotype",
                   "retina" : "feature_type",
                   "spc" : "feature_type"
                   }

# dictionary with num to downsample to (num of positive lncRNA for cluster_header annotation)
ds_dict = {"bm" : 79,
           "breast" : 64,
           "kidney" : 203,
           "liver" : 41,
           "lung" : 83,
           "m1" : 262,
           "mtg" : 386,
           "retina" : 703,
           "spc" : 703}

In [ ]:
### ARGS SETUP
sys.argv = [
    "", # placeholder for script name
    "data_spc_ds.h5ad", # name of data file
    "spc", # tissue abbreviation
    cluster_headers["spc"], # cluster header column in obs
    genetype_headers["spc"] # gene subset column in var
]

In [43]:
### CONFIGURATION -- Set the paths to the data folder and output folder
print_header("Configuring Environment")

data_folder = "../data_clean/" # path to folder containing the input data file (.h5ad format)
filename = sys.argv[1]
file = data_folder + filename

output_folder = f"../output_biowulf/{sys.argv[2]}/"
preprocessed_folder = "../data_preprocessed/"

seed = 0 # random seed for reproducibility
np.random.seed(seed) # set np seed
random.seed(seed) # set random seed

cluster_header = sys.argv[3]   # column name in adata.obs that contains the cluster labels
subset_col = sys.argv[4]       # column name in adata.var that contains the gene feature type
subset_gene = "protein_coding" # feature type to subset the data by (EX: "lncRNA" or "protein_coding")

print("Tissue:", sys.argv[2])
print("Cluster Header:", sys.argv[3])


CONFIGURING ENVIRONMENT
Tissue: spc
Cluster Header: PrimaryAnnotation


In [41]:
### IMPORT DATA -- Load the data and explore the dataset
print_header("Importing/Loading Data")
adata = sc.read_h5ad(file) # load the data into an AnnData object
print(adata)


IMPORTING/LOADING DATA
AnnData object with n_obs × n_vars = 28625 × 35467
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'PrimaryAnnotation', 'ThirdAnnotation', 'percent.mt', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_umap_HM_01'


In [8]:
### NORMALIZE/SCALE DATA -- Normalize and scale (log1p)
### cannot do this with retina data, already done + no raw counts
if adata.raw is not None and hasattr(adata.raw, "X"):
    print_header("Normalize/Scale Data")

    if sys.argv[2] in ["mtg", "m1"]:
        adata.raw._var.index = adata.raw.var["gene"]
        adata.X = adata.raw[:, adata.var_names].X.copy()
    else:
        adata.X = adata.raw.X.copy()

    sc.pp.normalize_total(adata, target_sum=1e4, inplace=True)
    sc.pp.log1p(adata)
    print("Data normalized and log1p transformed.")
else:
    print("No Normalizing/Scaling, data passed has no raw.X")


NORMALIZE/SCALE DATA
Data normalized and log1p transformed.


In [40]:
### CREATE GENE TYPE SUBSET -- Subset the data to only include genes of a specific feature type (EX: "lncRNA" or "protein_coding")
# adata_subset = adata[:, adata.var[subset_col] == subset_gene].copy()
# OR: compute subset_gene mask on the adata object
print_header(f"Create {subset_gene} Subset")
subset_mask = adata.var[subset_col].isin([subset_gene])
adata_subset = adata[:, subset_mask]
print(adata_subset)


CREATE PROTEIN_CODING SUBSET
View of AnnData object with n_obs × n_vars = 28625 × 19261
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'PrimaryAnnotation', 'ThirdAnnotation', 'percent.mt', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title', 'log1p'
    obsm: 'X_umap_HM_01'


## NS-Forest Runs

1. Run preprocessing steps on coding subset data (generate cluster medians and binary scores). Write preprocessed data.

2. Divide coding genes into ($n_{\text{coding genes}}$ / $n_{\text{downsample}}$) folds, each containing $n_{\text{downsample}}$ genes. Do this by selecting and storing indices of coding genes in each fold as a dict (so kvp "fold_{i}" : [list with indices])

3. For each fold...
    a. Extract the selected coding genes for that fold (in X, binary scores, and medians)
    b. Run NS-Forest on the fold.
    c. Extract performance results into dataframe with column fold = "fold_{i}"

4. Group by clusterName and summarize f_score, precision, recall, onTarget by average

5. Join to get software_version and clusterSize

6. Output averaged performance across folds per cluster.

In [10]:
# Preprocessing: Cluster Medians
print_header("Preprocess: Cluster Medians")
adata_subset = ns.pp.prep_medians(adata_subset, cluster_header)


PREPROCESS: CLUSTER MEDIANS


Calculating medians per cluster: 100%|██████████| 9/9 [00:05<00:00,  1.66it/s]
/opt/anaconda3/envs/nsforest/lib/python3.11/site-packages/nsforest/preprocessing/_add_ann.py:118: ImplicitModificationWarning: Setting element `.varm['medians_PrimaryAnnotation']` of view, initializing view as actual.
  adata.varm['medians_' + cluster_header] = cluster_medians


Saving medians as adata.varm.medians_PrimaryAnnotation
median: 0.0
mean: 0.071
std: 0.31
Only positive genes selected. 5114 positive genes out of 19261 total genes
--- 6.449984073638916 seconds ---


In [11]:
# Preprocessing: Binary Scores
print_header("Preprocess: Binary Score")
adata_subset = ns.pp.prep_binary_scores(adata_subset, cluster_header)


PREPROCESS: BINARY SCORE


Calculating binary scores per cluster: 100%|██████████| 9/9 [00:02<00:00,  3.61it/s]

Saving binary scores as adata.varm.binary_scores_PrimaryAnnotation
median: 0.0
mean: 0.179
std: 0.334
--- 2.5145888328552246 seconds ---


In [ ]:
## SAVE PREPROCESSED DATA AS NEW .h5ad
print_header("Preprocess: Saving Preprocessed Data")
    
filepp = filename.replace(".h5ad", f"_subset_{subset_gene}_preprocessed.h5ad")
print(f"Saving new anndata object as...\n{preprocessed_folder + filepp}")
adata_subset.write_h5ad(preprocessed_folder + filepp)


PREPROCESS: SAVING PREPROCESSED DATA
Saving new anndata object as...
../data_preprocessed/data_spc_ds_subset_protein_coding_preprocessed.h5ad


In [17]:
# DIVIDE CODING GENES INTO FOLDS
print_header("Divide Coding Genes into Folds")

# get number of folds
num_folds = min(int(adata_subset.varm[f'binary_scores_{cluster_header}'].shape[0] / ds_dict[sys.argv[2]]), 15) # limit
print((f"There are {adata_subset.varm[f'binary_scores_{cluster_header}'].shape[0]} positive coding genes "
       f"and {ds_dict[sys.argv[2]]} positive lncRNA genes for {cluster_header}. When downsampling to {ds_dict[sys.argv[2]]}, "
       f"we can create a maximum of {int(adata_subset.varm[f'binary_scores_{cluster_header}'].shape[0] / ds_dict[sys.argv[2]])} folds."))
print(f"We will divide the coding genes into {num_folds} folds.")

# build fold dictionary
fold_dict = {}
for i in range(num_folds):
    fold_dict[f"fold_{i+1}"] = random.sample(list(adata_subset.varm[f'binary_scores_{cluster_header}'].index), ds_dict[sys.argv[2]])


DIVIDE CODING GENES INTO FOLDS
There are 5114 positive coding genes and 703 positive lncRNA genes for PrimaryAnnotation. When downsampling to 703, we can create a maximum of 7 folds.
We will divide the coding genes into 7 folds.


In [27]:
# RUN NSFOREST -- Run NSForest on each fold of coding genes
print_header("Run NSForest on Each Fold")

results_list = [] # initialize empty list to store results
columns = ["clusterName", "f_score", "precision", "recall", "onTarget"] # columns to extract
silent_output = io.StringIO() # captures output from nsforest runs

for fold_name, coding_genes in fold_dict.items():
    print(f"Running NSForest on {fold_name}...")
    # subset adata to only include the genes in the current fold
    adata_fold = adata_subset[:, coding_genes].copy()
    print("- adata_fold:\n", adata_fold, "\n")
    
    # run NSForest on the current fold
    with warnings.catch_warnings(action="ignore"):
        with contextlib.redirect_stdout(silent_output):
            results_fold = ns.nsforesting.NSForest(adata_fold, cluster_header, save_supplementary = False, save = False)

    print(f"Completed NS-Forest run on {fold_name}. Extracting results...\n")
    # extract clusterName, f_score, precision, recall, and onTarget from results_fold and add to results_df
    results_fold_df = pd.DataFrame(results_fold)
    results_fold_df["fold"] = fold_name
    print("results_fold_df:\n", results_fold_df, "\n")
    results_list.append(results_fold_df)
    print("Length of results_list:", len(results_list))
    if fold_name == "fold_2":
        break
    


RUN NSFOREST ON EACH FOLD
Running NSForest on fold_1...
- adata_fold:
 AnnData object with n_obs × n_vars = 28625 × 703
    obs: 'donor_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'PrimaryAnnotation', 'ThirdAnnotation', 'percent.mt', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'is_pre_analysis', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title', 'log1p'
    obsm: 'X_umap_HM_01'
    varm: 'medians_PrimaryAnnotation', 'binary_scores_PrimaryAnnotation' 



Calculating medians per cluster: 100%|██████████| 9/9 [00:00<00:00, 31.54it/s]


Completed NS-Forest run on fold_1. Extracting results...

results_fold_df:
   software_version     cluster_header clusterName  clusterSize   f_score  \
0              4.1  PrimaryAnnotation       Astro         4430  0.799670   
1              4.1  PrimaryAnnotation        Endo          540  0.339018   
2              4.1  PrimaryAnnotation   Ependymal           68  0.816327   
3              4.1  PrimaryAnnotation  Lymphocyte          163  0.802998   
4              4.1  PrimaryAnnotation         MNs           58  0.903361   
5              4.1  PrimaryAnnotation       Micro         2651  0.961621   
6              4.1  PrimaryAnnotation      Neuron         1900  0.929027   
7              4.1  PrimaryAnnotation       Oligo        16840  0.918101   
8              4.1  PrimaryAnnotation         Opc         1975  0.843684   

   precision    recall     TN   FP    FN     TP  marker_count  \
0   0.865519  0.613093  23773  422  1714   2716             1   
1   0.349785  0.301852  27782  30

Calculating medians per cluster: 100%|██████████| 9/9 [00:00<00:00, 30.39it/s]

Completed NS-Forest run on fold_2. Extracting results...

results_fold_df:
   software_version     cluster_header clusterName  clusterSize   f_score  \
0              4.1  PrimaryAnnotation       Astro         4430  0.915606   
1              4.1  PrimaryAnnotation        Endo          540  0.631757   
2              4.1  PrimaryAnnotation   Ependymal           68  0.793919   
3              4.1  PrimaryAnnotation  Lymphocyte          163  0.612245   
4              4.1  PrimaryAnnotation         MNs           58  0.903361   
5              4.1  PrimaryAnnotation       Micro         2651  0.895914   
6              4.1  PrimaryAnnotation      Neuron         1900  0.908213   
7              4.1  PrimaryAnnotation       Oligo        16840  0.915255   
8              4.1  PrimaryAnnotation         Opc         1975  0.812551   

   precision    recall     TN   FP    FN     TP  marker_count  \
0   0.994509  0.695034  24178   17  1351   3079             2   
1   0.795745  0.346296  28037   4

In [35]:
# CONCATENATE/AGG RESULTS -- Obtain average performance measures across folds
print_header("Concatenating/Aggregating Results")

results_df = pd.concat(results_list)

results_agg = results_df.groupby(['clusterName']).agg(
    avg_precision=pd.NamedAgg(column = "precision", aggfunc="mean"),
    avg_recall=pd.NamedAgg(column = "recall", aggfunc="mean"),
    avg_f_score=pd.NamedAgg(column = "f_score", aggfunc="mean"),
    avg_onTarget=pd.NamedAgg(column = "onTarget", aggfunc="mean"),
).reset_index(inplace=False)

print("First 10 rows of aggregated Results:")
print(results_agg.head(10))


CONCATENATING/AGGREGATING RESULTS
First 10 rows of aggregated Results:
  clusterName  avg_precision  avg_recall  avg_f_score  avg_onTarget
0       Astro       0.930014    0.654063     0.857638           1.0
1        Endo       0.572765    0.324074     0.485388           1.0
2   Ependymal       0.912281    0.580882     0.805123           1.0
3  Lymphocyte       0.960088    0.358896     0.707621           1.0
4         MNs       0.955556    0.741379     0.903361           1.0
5       Micro       0.969625    0.795549     0.928768           1.0
6      Neuron       0.978068    0.739211     0.918620           1.0
7       Oligo       0.968089    0.756265     0.916678           1.0
8         Opc       0.974592    0.517215     0.828118           1.0


In [39]:
## SAVE FULL AND AGGREGATED RESULTS
print_header("Saving Full and Aggregated Results")
    
# full results
filefull = f"dscoding_results_full_{sys.argv[2]}.csv"
print(f"Saving full results as...\n{output_folder + filefull}")
results_df.to_csv(output_folder + filefull, index=False)

# full results
fileagg = f"dscoding_results_agg_{sys.argv[2]}.csv"
print(f"Saving full results as...\n{output_folder + fileagg}")
results_agg.to_csv(output_folder + fileagg, index=False)


SAVING FULL AND AGGREGATED RESULTS
Saving full results as...
../output_biowulf/spc/dscoding_results_full_spc.csv
Saving full results as...
../output_biowulf/spc/dscoding_results_agg_spc.csv
